# Treino experimental da voz da Lia (Piper Plus / VITS)

**Notebook preparado para execução manual no Google Colab.** Ao executar o passo de upload, os arquivos de voz serão transferidos do seu computador para o runtime Google. Você autorizou esse envio nesta conversa. O notebook não monta Drive, não publica no Hugging Face e não envia áudio para terceiros além do ambiente Colab escolhido. O runtime é temporário; interrompa e apague a sessão ao terminar.

## Condição de licença

Candidato técnico: `ayousanz/piper-plus-base` (Piper Plus multilingual, inclui português). O model card publica dados/receita, mas não declara claramente a licença dos pesos. O código do Piper Plus se declara MIT, o que **não licencia os pesos**. A autorização dada nesta conversa permite a prova privada/local; o fluxo prossegue sem edição de código. Não publique nem redistribua o checkpoint/base/fine-tune sem confirmar os direitos.

Não use áudio de outra pessoa sem autorização. Revise transcrições manualmente. Este fluxo não faz ASR. Mantenha originais privados fora do Git.

**Entrada:** pode ser o MP3 original, sem você montar ZIP ou CSV. O notebook cria os segmentos WAV e transcreve para um rascunho de `metadata.csv`; você só confere a tabela e confirma. Também aceita um ZIP pronto com `metadata.csv` (`id|texto`) e WAVs mono.

In [ ]:
# @title Aceitar avaliação privada (automático)
# Controles explícitos. Nenhum upload ocorre nesta célula.
PRIVATE_EVALUATION_ONLY = True
ALLOW_UNRESOLVED_BASE_WEIGHTS_FOR_PRIVATE_TEST = True  # autorização do usuário: avaliar em privado; sem redistribuição

if not PRIVATE_EVALUATION_ONLY or not ALLOW_UNRESOLVED_BASE_WEIGHTS_FOR_PRIVATE_TEST:
    raise RuntimeError('Este fluxo foi autorizado apenas para avaliação privada, sem redistribuição.')
print('Modo privado autorizado. Nenhum áudio foi enviado ainda; o upload ocorre na célula própria.')

## 1. Ambiente do treino

Instalações ocorrem apenas dentro do Colab, não no Windows nem no repositório. Código fixado no commit consultado em 2026-09-30. **Antes de executar, o notebook solicita GPU T4 automaticamente nos metadados**; o Colab ainda pode oferecer outra GPU ou nenhuma, conforme disponibilidade. Se o runtime iniciar sem GPU, escolha T4 no menu `Runtime > Change runtime type > Hardware accelerator`. Depois use `Runtime > Run all`: ele instala o ambiente, pede o ZIP uma vez, valida os dados, treina, exporta e oferece o resultado para download.

In [ ]:
# @title Preparar Colab com GPU T4 e instalar dependências
import pathlib, subprocess, torch, sys, shutil
# Usa a API do PyTorch; não depende do executável nvidia-smi.
if not torch.cuda.is_available():
    raise RuntimeError('Este runtime Colab não tem GPU disponível. Antes de executar novamente, selecione Runtime > Change runtime type > Hardware accelerator > GPU e reinicie a sessão.')
print('GPU:', torch.cuda.get_device_name(0), '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
# Piper-Plus documents espeak-ng as a preprocessing prerequisite.
if not shutil.which('espeak-ng'):
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','-qq','espeak-ng'], check=True)
print('espeak-ng:', shutil.which('espeak-ng'))
REPO = pathlib.Path('/content/piper-plus')
PIN = '03f22b895d6b87d3733223c0844cb2ef6569937b'
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/ayutaz/piper-plus.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'checkout',PIN], check=True)
subprocess.run(['python','-m','pip','install','uv'], check=True)
subprocess.run(['uv','pip','install','--system','.[train]'], cwd=REPO, check=True)
subprocess.run([sys.executable,'-m','pip','install','huggingface_hub','faster-whisper==1.2.1','av==18.1.0','soundfile'], check=True)
# Confirma no mesmo interpretador antes de liberar a célula de upload.
subprocess.run([sys.executable,'-c','import av, faster_whisper, soundfile; print("PyAV", av.__version__, "Faster-Whisper ready")'], check=True)
print('Commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())


## 2. Enviar áudio MP3 ou dataset pronto

Escolha seu MP3 diretamente no seletor — não precisa preparar ZIP, WAV ou CSV. O notebook usa Faster-Whisper `small` em CPU, com dependências PyAV instaladas/verificadas antes do upload e pesos MIT pinados, dentro do próprio runtime Colab (sem serviço externo), divide o áudio em segmentos e cria WAVs mono + `metadata.csv`. Se já tiver dataset pronto, também aceita um ZIP com `metadata.csv` e WAVs. O reconhecimento cria rascunhos: confira a tabela antes de confirmar o treino.


In [ ]:
# @title Enviar MP3 e gerar segmentos/transcrição automaticamente
from google.colab import files
from pathlib import Path
import zipfile, shutil, csv, re, subprocess, sys, importlib.util

DATA_ROOT = Path('/content/lia_voice_data')
shutil.rmtree(DATA_ROOT, ignore_errors=True)
DATA_ROOT.mkdir(parents=True)
uploaded = files.upload()  # escolha o MP3; o áudio é enviado somente ao Colab
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
audio_names = [name for name in uploaded if Path(name).suffix.lower() in {'.mp3','.wav','.m4a','.flac','.ogg'}]
if zip_names:
    if len(zip_names) != 1 or audio_names:
        raise ValueError('Envie um ZIP pronto, ou um/mais arquivos de áudio, não os dois.')
    with zipfile.ZipFile(zip_names[0]) as zf:
        root = DATA_ROOT.resolve()
        for info in zf.infolist():
            target = (DATA_ROOT / info.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError('Caminho inseguro dentro do ZIP; extração bloqueada.')
        zf.extractall(DATA_ROOT)
    print('Dataset extraído; seguirá para validação de metadata.csv.')
elif audio_names and len(audio_names) == len(uploaded):
    # Self-heal if this cell is run without first running the setup cell.
    missing = [name for name in ('av','faster_whisper','soundfile') if importlib.util.find_spec(name) is None]
    if missing:
        print('Instalando dependências de áudio ausentes:', ', '.join(missing))
        subprocess.run([sys.executable,'-m','pip','install','faster-whisper==1.2.1','av==18.1.0','soundfile'], check=True)
    import av, numpy as np, soundfile as sf
    from faster_whisper import WhisperModel
    audio_root = DATA_ROOT
    metadata_rows = []
    from huggingface_hub import snapshot_download
    ASR_MODEL_REVISION='536b0662742c02347bc0e980a01041f333bce120'
    ASR_MODEL_PATH=snapshot_download('Systran/faster-whisper-small', revision=ASR_MODEL_REVISION)
    asr = WhisperModel(ASR_MODEL_PATH, device='cpu', compute_type='int8')
    def decode_mono_22050(path):
        chunks=[]
        with av.open(str(path)) as container:
            stream=container.streams.audio[0]
            resampler=av.AudioResampler(format='flt', layout='mono', rate=22050)
            for frame in container.decode(stream):
                for converted in resampler.resample(frame):
                    chunks.append(converted.to_ndarray().reshape(-1).astype(np.float32))
            for converted in resampler.resample(None):
                chunks.append(converted.to_ndarray().reshape(-1).astype(np.float32))
        if not chunks: raise ValueError(f'Não foi possível decodificar {path.name}')
        return np.concatenate(chunks)
    counter=0
    for original_name in audio_names:
        audio_path=Path('/content')/original_name
        print('Transcrevendo em português:', original_name)
        segments, info = asr.transcribe(str(audio_path), language='pt', beam_size=5,
                                        vad_filter=True, word_timestamps=True)
        segments=list(segments)  # consome o gerador e conclui a transcrição
        waveform=decode_mono_22050(audio_path)
        sr=22050
        words=[]
        for segment in segments:
            if segment.words:
                words.extend(segment.words)
            elif segment.text.strip():
                words.append(type('Word',(),{'start':segment.start,'end':segment.end,'word':segment.text})())
        groups=[]; current=[]; group_start=None
        for word in words:
            text=(word.word or '').strip()
            if not text: continue
            if group_start is None: group_start=float(word.start)
            current.append(word)
            elapsed=float(word.end)-group_start
            if text.endswith(('.', '?', '!', '…')) or elapsed >= 8.0:
                groups.append(current); current=[]; group_start=None
        if current: groups.append(current)
        for group in groups:
            start=max(0.0,float(group[0].start)-0.08)
            end=min(len(waveform)/sr,float(group[-1].end)+0.12)
            text=''.join(w.word or '' for w in group).strip().replace('|',',').replace('\n',' ')
            if end-start < 0.8 or not text: continue
            counter+=1
            clip_id=f'fala_{counter:04d}'
            i0=max(0,int(start*sr)); i1=min(len(waveform),int(end*sr))
            sf.write(audio_root/(clip_id+'.wav'),waveform[i0:i1],sr,subtype='PCM_16')
            metadata_rows.append((clip_id,text))
        print(f"Idioma detectado: {info.language}; confiança {info.language_probability:.2f}")
    if not metadata_rows:
        raise ValueError('A transcrição não gerou segmentos utilizáveis.')
    metadata_path=DATA_ROOT/'metadata.csv'
    with metadata_path.open('w',encoding='utf-8',newline='') as f:
        writer=csv.writer(f,delimiter='|',lineterminator='\n',quoting=csv.QUOTE_MINIMAL)
        writer.writerows(metadata_rows)
    print(f'Criados {len(metadata_rows)} segmentos e metadata.csv em {DATA_ROOT}.')
else:
    raise ValueError('Envie um MP3/áudio, ou um ZIP pronto com metadata.csv e WAVs.')


## 3. Validar estrutura e áudio

`metadata.csv` usa pipe (`id|texto`), sem cabeçalho. Cada identificador precisa corresponder a `id.wav`. O pré-processamento Piper converte para 22.050 Hz. Revise transcrições, nomes, números, clipping, ruído e cortes antes do treino. A célula mostra a tabela dos rascunhos e pausa para você digitar `SIM`; se quiser corrigir algum texto, edite `/content/lia_voice_data/metadata.csv` no painel Files antes de confirmar. Essa é uma revisão de texto, não configuração de código.

In [ ]:
# @title Revisar transcrição no campo de texto e confirmar
import soundfile as sf
import zipfile
DATA_ROOT = Path('/content/lia_voice_data')
meta_files = list(DATA_ROOT.rglob('metadata.csv'))
if len(meta_files) != 1:
    raise ValueError(f'Esperado um metadata.csv; encontrados {len(meta_files)}')
META = meta_files[0]
AUDIO_DIR = META.parent
rows, missing, durations, rates = [], [], [], set()
for line_no, line in enumerate(META.read_text(encoding='utf-8').splitlines(), 1):
    if not line.strip() or line.lstrip().startswith('#'): continue
    parts = line.split('|', 1)
    if len(parts) != 2 or not all(part.strip() for part in parts):
        raise ValueError(f'Linha {line_no} inválida; esperado id|texto')
    key, text = parts[0].strip(), parts[1].strip()
    wav = AUDIO_DIR / (key + '.wav')
    if not wav.is_file():
        missing.append((line_no, key)); continue
    info = sf.info(wav)
    if info.channels != 1 or info.frames == 0:
        raise ValueError(f'{wav.name}: esperado WAV mono não vazio')
    rows.append((key, text)); durations.append(info.duration); rates.add(info.samplerate)
if missing: raise FileNotFoundError(f'Áudios ausentes: {missing[:10]}')
if not rows: raise ValueError('Dataset vazio')
print(f'{len(rows)} frases; {sum(durations)/60:.1f} min; duração {min(durations):.2f}-{max(durations):.2f} s')
print('Sample rates encontrados:', sorted(rates))
print('Validação estrutural concluída. Rascunho de transcrições:')
from IPython.display import display
from google.colab import output, files
import ipywidgets as widgets
output.enable_custom_widget_manager()
print(f'{len(rows)} segmentos. Revise/edite o texto abaixo; somente as transcrições serão alteradas.')
editor=widgets.Textarea(value=META.read_text(encoding='utf-8'), description='metadata.csv',
                        layout=widgets.Layout(width='100%',height='420px'),
                        style={'description_width':'initial'})
display(editor)
approval=input('Após revisar o campo acima, digite SIM para salvar e continuar. Qualquer outra resposta interrompe: ').strip().upper()
if approval != 'SIM':
    raise RuntimeError('Parei antes do treino. Reexecute as células após revisar os dados.')
new_rows=[]
for line_no,line in enumerate(editor.value.splitlines(),1):
    if not line.strip(): continue
    parts=line.split('|',1)
    if len(parts)!=2 or not parts[0].strip() or not parts[1].strip():
        raise ValueError(f'Linha {line_no} inválida; preserve o formato id|texto.')
    new_rows.append((parts[0].strip(),parts[1].strip()))
expected_ids={row[0] for row in rows}
new_ids=[row[0] for row in new_rows]
if len(new_ids)!=len(set(new_ids)) or set(new_ids)!=expected_ids:
    raise ValueError('Não altere os IDs das gravações; corrija somente o texto transcrito.')
with META.open('w',encoding='utf-8',newline='') as f:
    for clip_id,text in new_rows:
        f.write(f'{clip_id}|{text.replace(chr(10)," " )}\n')
print('Transcrições salvas; seguindo para pré-processamento e treino.')
# Archive only the reviewed source WAVs and metadata (not models/cache), so it
# can be downloaded and re-uploaded after Colab resets without repeating ASR.
VOICE_DATA_ARCHIVE = Path('/content/lia_voice_dataset.zip')
with zipfile.ZipFile(VOICE_DATA_ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as zf:
    zf.write(META,arcname=META.relative_to(DATA_ROOT))
    for clip_id,_text in new_rows:
        wav = AUDIO_DIR / (clip_id + '.wav')
        if not wav.is_file():
            raise FileNotFoundError(f'Não foi possível empacotar o áudio revisado: {wav}')
        zf.write(wav,arcname=wav.relative_to(DATA_ROOT))
print(f'Backup ZIP pronto: {VOICE_DATA_ARCHIVE} ({VOICE_DATA_ARCHIVE.stat().st_size/1024/1024:.1f} MiB). Baixando para reutilizar após reset do Colab.')
files.download(str(VOICE_DATA_ARCHIVE))

## 4. Localizar base, revisar licença e preparar

O notebook baixa automaticamente o checkpoint e `config.json` da revisão pinada do model card. Verifica SHA-256 e tamanho do checkpoint. O arquivo usa formato PyTorch checkpoint/pickle: só confie no artefato e no código upstream fixados. A licença do peso não está clara; conforme sua autorização, o uso fica restrito a avaliação privada, sem publicação ou redistribuição.

In [ ]:
# @title Baixar e verificar a base (automático)
from huggingface_hub import hf_hub_download
import hashlib, json, shutil, subprocess, sys
from pathlib import Path
BASE_REPO = 'ayousanz/piper-plus-base'
BASE_REVISION = 'ce006e5da7851fdc469887026eeb3f2175e7e080'
BASE_CHECKPOINT = hf_hub_download(BASE_REPO, 'model.ckpt', revision=BASE_REVISION)
BASE_CONFIG = hf_hub_download(BASE_REPO, 'config.json', revision=BASE_REVISION)
EXPECTED_SHA256 = '83c079abc50575eeb837e1d21e98aefafb23981b9f2cbee247cbcd222df6216b'
sha = hashlib.sha256(Path(BASE_CHECKPOINT).read_bytes()).hexdigest()
if sha != EXPECTED_SHA256:
    raise ValueError(f'Hash inesperado do checkpoint: {sha}')
DATASET_DIR = '/content/lia_dataset'
OUT_DIR = '/content/lia_voice_experiment'
print('Checkpoint verificado:', sha, '| bytes:', Path(BASE_CHECKPOINT).stat().st_size)
base_config = json.loads(Path(BASE_CONFIG).read_text(encoding='utf-8'))
PREPROCESS_LANGUAGE = base_config.get('language', {}).get('code', '')
if not PREPROCESS_LANGUAGE or 'pt' not in PREPROCESS_LANGUAGE.split('-'):
    raise ValueError(f'O checkpoint não declara português no inventário de línguas: {PREPROCESS_LANGUAGE!r}')
# Piper-Plus otherwise routes Latin script to English whenever 'en' is in the
# six-language model. Apply this patch here, every time, so it cannot be skipped
# when resuming a Colab session or running only the preprocessing cell.
PREPROCESS_SOURCE = Path(REPO) / 'src/python/piper_train/preprocess.py'
# IMPORTANT: `python -m piper_train.preprocess` resolves the installed module,
# which in Colab is normally under site-packages (as the observed traceback
# confirmed), not the checkout's src/python file. Patch both copies and verify
# the exact module path the child interpreter will execute.
INSTALLED_PREPROCESS_SOURCE = Path(subprocess.check_output(
    [sys.executable,'-c','import piper_train.preprocess as p; print(p.__file__)'],
    text=True
).strip()).resolve()
PT_PATCH_MARKER = '# Arena notebook: default Latin G2P to Portuguese for PT fine-tuning.'
def patch_preprocess_language(path):
    source = path.read_text(encoding='utf-8')
    if PT_PATCH_MARKER in source:
        return
    old_phonemizer = '    phonemizer = MultilingualPhonemizer(lang_parts)\n'
    new_phonemizer = (PT_PATCH_MARKER + '\n'
        '    default_latin_language = ("pt" if "pt" in lang_parts else "en" if "en" in lang_parts else lang_parts[0])\n'
        '    phonemizer = MultilingualPhonemizer(lang_parts, default_latin_language=default_latin_language)\n')
    old_detector = ('                default_latin_language="en"\n'
        '                if "en" in args.lang_parts\n'
        '                else args.lang_parts[0],\n')
    new_detector = ('                default_latin_language="pt"\n'
        '                if "pt" in args.lang_parts\n'
        '                else ("en" if "en" in args.lang_parts else args.lang_parts[0]),\n')
    if source.count(old_phonemizer) != 1 or source.count(old_detector) != 1:
        raise RuntimeError(f'{path} divergiu do preprocess.py pinado; ajuste automático cancelado.')
    source = source.replace(old_phonemizer,new_phonemizer).replace(old_detector,new_detector)
    path.write_text(source,encoding='utf-8')
for path in {PREPROCESS_SOURCE.resolve(), INSTALLED_PREPROCESS_SOURCE}:
    patch_preprocess_language(path)
    executed_source = path.read_text(encoding='utf-8')
    if PT_PATCH_MARKER not in executed_source or 'default_latin_language=default_latin_language' not in executed_source:
        raise RuntimeError(f'Patch PT não confirmado no módulo {path}.')
print('Módulos PT preparados:')
print(' - checkout:', PREPROCESS_SOURCE.resolve())
print(' - python -m executará:', INSTALLED_PREPROCESS_SOURCE)
G2P_SOURCE = Path(REPO) / 'src/python/g2p'
if str(G2P_SOURCE) not in sys.path:
    sys.path.insert(0,str(G2P_SOURCE))
from piper_plus_g2p.multilingual import MultilingualPhonemizer
pt_phonemizer = MultilingualPhonemizer(
    ['ja','en','zh','es','fr','pt'], default_latin_language='pt'
)
pt_tokens,_ = pt_phonemizer.phonemize_with_prosody('Olá, este é um teste em português.')
if not pt_tokens:
    raise RuntimeError('O fonemizador português não produziu fonemas; o pré-processamento não será iniciado.')
print('G2P PT verificado:', ' '.join(pt_tokens[:12]), '| idioma padrão: pt | mapa de 6 idiomas preservado')
if shutil.which('espeak-ng') is None:
    print('Instalando espeak-ng, dependência documentada para preprocessing...')
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','-qq','espeak-ng'], check=True)
if len(rows) < 2:
    raise ValueError('O pré-processamento do Piper-Plus requer pelo menos 2 frases; verifique metadata.csv e os WAVs.')
# Piper-Plus computes batch_size as utterances/(workers*2); cap workers so a
# small Colab dataset cannot produce batch_size=0, and avoid over-forking CPUs.
PREPROCESS_WORKERS = max(1, min(4, len(rows) // 2))
# Remove partial output from a previous failed attempt; preprocess appends JSONL.
shutil.rmtree(DATASET_DIR, ignore_errors=True)
preprocess_cmd = [
    sys.executable,'-u','-m','piper_train.preprocess',
    '--input-dir',str(AUDIO_DIR),'--output-dir',DATASET_DIR,
    '--language',PREPROCESS_LANGUAGE,'--dataset-format','ljspeech','--sample-rate','22050',
    '--single-speaker','--phoneme-type','multilingual','--max-workers',str(PREPROCESS_WORKERS)
]
print('Línguas do checkpoint:', PREPROCESS_LANGUAGE)
print('Frases:',len(rows),'| workers:',PREPROCESS_WORKERS)
print('Pré-processamento proposto:', ' '.join(preprocess_cmd))
PREPROCESS_LOG = Path('/content/lia_preprocess.log')

def run_preprocess(command, label):
    result = subprocess.run(command, cwd=REPO, text=True, capture_output=True)
    with PREPROCESS_LOG.open('a',encoding='utf-8') as log:
        log.write(f'\n===== {label} | exit={result.returncode} =====\n')
        log.write(result.stdout or '')
        log.write(result.stderr or '')
    if result.stdout:
        print(result.stdout, end='' if result.stdout.endswith('\n') else '\n')
    if result.stderr:
        print(result.stderr, end='' if result.stderr.endswith('\n') else '\n')
    if result.returncode:
        raise RuntimeError(f'Pré-processamento terminou com código {result.returncode}; consulte também {PREPROCESS_LOG}.')
    manifest = Path(DATASET_DIR) / 'dataset.jsonl'
    return sum(1 for line in manifest.open(encoding='utf-8') if line.strip()) if manifest.is_file() else 0

RUN_PREPROCESS = True
if RUN_PREPROCESS:
    PREPROCESS_LOG.write_text('Diagnóstico local do pré-processamento Piper-Plus\n',encoding='utf-8')
    processed = run_preprocess(preprocess_cmd,'tentativa normal')
    if processed == 0:
        # The pinned Piper-Plus worker suppresses worker stderr unless --debug is set.
        # Retry with one worker so the original per-utterance exception is visible.
        print('O Piper-Plus terminou sem exemplos. Repetindo automaticamente com --debug e 1 worker para revelar o erro interno...')
        shutil.rmtree(DATASET_DIR,ignore_errors=True)
        debug_cmd = list(preprocess_cmd)
        debug_cmd[debug_cmd.index('--max-workers')+1] = '1'
        debug_cmd.append('--debug')
        processed = run_preprocess(debug_cmd,'tentativa diagnóstica --debug')
    if processed == 0:
        raise RuntimeError(f'O pré-processamento não gerou nenhum exemplo. Veja o erro detalhado acima e o log {PREPROCESS_LOG}; o treino foi interrompido antes de iniciar.')
    print(f'Pré-processamento validado: {processed} exemplo(s) no dataset.jsonl.')
    if processed < len(rows):
        print(f'Aviso: {len(rows)-processed} de {len(rows)} frases foram ignoradas; consulte os avisos do Piper-Plus acima.')
else:
    print('Pré-processamento não iniciado.')

## 5. Treino

A receita segue os parâmetros publicados para Piper Plus multilingual, com 500 épocas e checkpoints de retomada a cada 25 épocas. O treino inicia automaticamente após upload e validações. A duração depende do tamanho do corpus e da GPU alocada; o Colab pode interromper sessões gratuitas.

In [ ]:
# @title Treinar a voz (automático)
import json, os, subprocess, sys
from pathlib import Path

train_config_path = Path(DATASET_DIR) / 'config.json'
train_manifest_path = Path(DATASET_DIR) / 'dataset.jsonl'
if not train_config_path.is_file() or not train_manifest_path.is_file():
    raise FileNotFoundError(f'Dataset pré-processado incompleto: esperado {train_config_path} e {train_manifest_path}')
train_config = json.loads(train_config_path.read_text(encoding='utf-8'))
train_examples = sum(1 for line in train_manifest_path.open(encoding='utf-8') if line.strip())
if train_examples < 2:
    raise ValueError(f'Dataset de treino insuficiente: {train_examples} exemplos em {train_manifest_path}. Reexecute a célula de pré-processamento; ela agora repete automaticamente com --debug e salva o diagnóstico em /content/lia_preprocess.log.')
print(f"Dataset: {train_examples} exemplos | speakers={train_config.get('num_speakers')} | languages={train_config.get('num_languages')} | symbols={train_config.get('num_symbols')}")
print('Python:',sys.version.split()[0],'| PyTorch:',torch.__version__,'| CUDA:',torch.version.cuda)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'indisponível')

train_cmd = [
    sys.executable,'-u','-m','piper_train','--dataset-dir',DATASET_DIR,
    '--prosody-dim','16','--accelerator','gpu','--devices','1',
    '--precision','32-true','--max_epochs','500','--batch-size','4',
    '--samples-per-speaker','4','--checkpoint-epochs','25',
    '--base_lr','2e-5','--disable_auto_lr_scaling','--ema-decay','0.9995',
    '--max-phoneme-ids','400','--no-wavlm','--no-compile','--num-workers','2',
    '--resume-from-multispeaker-checkpoint',BASE_CHECKPOINT,
    '--default_root_dir',OUT_DIR
]
print('Comando piloto:', ' '.join(train_cmd))
RUN_TRAINING = True
if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError('Este Colab está sem GPU NVIDIA. Selecione Runtime > Change runtime type > GPU, reconecte e execute as células novamente.')
    # Sem login nem envio de métricas a serviço externo (modo de avaliação privada).
    # TensorBoard continua disponível localmente em OUT_DIR/lightning_logs.
    train_env = os.environ.copy()
    train_env.update({'WANDB_MODE':'disabled','PYTHONUNBUFFERED':'1'})
    train_log = Path(OUT_DIR) / 'training.log'
    Path(OUT_DIR).mkdir(parents=True,exist_ok=True)
    print('W&B desativado; saída completa também será salva em:',train_log)
    with train_log.open('w',encoding='utf-8') as log_file:
        process = subprocess.Popen(
            train_cmd,cwd=REPO,env=train_env,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line,end='',flush=True)
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
    if return_code:
        tail=''.join(train_log.read_text(encoding='utf-8',errors='replace').splitlines(keepends=True)[-80:])
        raise RuntimeError(f'Treino terminou com código {return_code}. Últimas linhas do log:\n{tail}\nLog completo: {train_log}')
else:
    print('Treino não iniciado.')


## 6. Exportar, empacotar e baixar

Depois que o treino terminar, a próxima célula encontra o checkpoint mais recente, exporta ONNX, inclui o JSON de configuração e abre o download do pacote. Nenhum arquivo é enviado/publicado automaticamente. Guarde o ZIP fora do repositório; a licença dos pesos segue sem esclarecimento, então o uso continua privado.

Ao terminar, interrompa e exclua o runtime Colab. O notebook não monta Drive.


In [ ]:
# @title Exportar e baixar a voz treinada (automático)
from pathlib import Path
import shutil, subprocess, zipfile
from google.colab import files

ckpts = [p for p in Path(OUT_DIR).rglob('*.ckpt') if p.is_file()]
if not ckpts:
    raise FileNotFoundError('Treino finalizado, mas não encontrei checkpoint .ckpt em '+OUT_DIR)
latest = max(ckpts, key=lambda p: p.stat().st_mtime)
EXPORT_DIR = Path(OUT_DIR) / 'export'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
ONNX = EXPORT_DIR / 'lia_voice.onnx'
export_cmd = ['python','-m','piper_train.export_onnx','--unify-emb-lang',str(latest),str(ONNX)]
print('Checkpoint selecionado:', latest)
subprocess.run(export_cmd, cwd=REPO, check=True)

# O exportador normalmente cria o JSON pareado. Se não o fizer, usar o config
# da mesma base pinada, cujo inventário de fonemas/arquitetura foi validado.
config_candidates = [Path(str(ONNX)+'.json'), ONNX.with_suffix('.onnx.json'), EXPORT_DIR/'config.json']
config_out = next((x for x in config_candidates if x.is_file()), None)
if config_out is None:
    config_out = Path(str(ONNX)+'.json')
    shutil.copy2(BASE_CONFIG, config_out)

package = Path('/content/lia_voice_private_eval.zip')
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as zf:
    zf.write(ONNX, ONNX.name)
    zf.write(config_out, 'lia_voice.onnx.json')
print('Pacote pronto:', package, '| bytes:', package.stat().st_size)
files.download(str(package))
